# Validação do Espaço Semântico e Assinatura de Impacto

Este notebook aborda as seguintes questões de pesquisa:

**RQ 1.1:** O espaço semântico gerado pelos embeddings captura de forma coerente as afiliações temáticas formais dos pesquisadores (áreas da CAPES/SBC, linhas de pesquisa dos PPGs)?

**RQ 1.2:** Existe uma "assinatura de impacto"? Ou seja, pesquisadores com maior impacto (e.g., índice-H, contagem de citações) estão localizados em regiões específicas do espaço semântico (e.g., no centro de clusters, em sobreposições)?


## 1. Carregar Dados e Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import umap
import hdbscan
import plotly.express as px
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
import plotly.io as pio

pio.templates.default = "plotly_white"

# Carregar os dados com features
try:
    df = pd.read_parquet('../data/processed/featured_authors.parquet')
    # Converter a coluna de embedding de lista para array numpy
    embeddings = np.array(df['embedding'].tolist())
except FileNotFoundError:
    print("Arquivo 'featured_authors.parquet' não encontrado. Certifique-se de que a Fase 2 foi executada.")
    df = pd.DataFrame() # Cria um dataframe vazio para evitar erros posteriores
    embeddings = np.array([])

## 2. Análise de Cluster e Redução de Dimensionalidade (RQ 1.1)

In [ ]:
if not df.empty:
    # Redução de dimensionalidade com UMAP
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
    embedding_2d = reducer.fit_transform(embeddings)
    df['umap_x'] = embedding_2d[:, 0]
    df['umap_y'] = embedding_2d[:, 1]
    
    # Clustering com HDBSCAN
    clusterer = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5)
    df['hdbscan_cluster'] = clusterer.fit_predict(embeddings)
    
    print(f"Número de clusters encontrados: {len(np.unique(df['hdbscan_cluster'])) - 1}")
    print(f"Número de outliers: {np.sum(df['hdbscan_cluster'] == -1)}")

### 2.1 Visualização dos Clusters

In [ ]:
if not df.empty:
    fig = px.scatter(
        df,
        x='umap_x',
        y='umap_y',
        color='hdbscan_cluster',
        hover_data=['display_name', 'institution_name_acronym', 'hdbscan_cluster'],
        title='Clusters de Pesquisadores (HDBSCAN) no Espaço Semântico (UMAP)',
        color_continuous_scale='viridis' # Ajuste a escala de cor se necessário
    )
    fig.show()

### 2.2 Comparação com Áreas da CAPES/SBC

In [ ]:
if not df.empty:
    # Supondo que a coluna 'gp_name' representa a área CAPES/SBC
    df_filtered = df[df['hdbscan_cluster'] != -1]
    
    nmi = normalized_mutual_info_score(df_filtered['gp_name'], df_filtered['hdbscan_cluster'])
    ari = adjusted_rand_score(df_filtered['gp_name'], df_filtered['hdbscan_cluster'])
    
    print(f"Normalized Mutual Information (NMI): {nmi:.4f}")
    print(f"Adjusted Rand Index (ARI): {ari:.4f}")
    
    fig = px.scatter(
        df,
        x='umap_x',
        y='umap_y',
        color='gp_name',
        hover_data=['display_name', 'institution_name_acronym'],
        title='Áreas Formais no Espaço Semântico (UMAP)'
    )
    fig.show()

## 3. Análise da Assinatura de Impacto (RQ 1.2)

In [ ]:
if not df.empty:
    # Calcular centroides dos clusters
    centroids = df_filtered.groupby('hdbscan_cluster')[['umap_x', 'umap_y']].mean().reset_index()
    centroids.rename(columns={'umap_x': 'centroid_x', 'umap_y': 'centroid_y'}, inplace=True)
    
    df_filtered = pd.merge(df_filtered, centroids, on='hdbscan_cluster')
    
    # Calcular distância euclidiana para o centroide
    df_filtered['dist_to_centroid'] = np.sqrt(
        (df_filtered['umap_x'] - df_filtered['centroid_x'])**2 + 
        (df_filtered['umap_y'] - df_filtered['centroid_y'])**2
    )
    
    fig = px.scatter(
        df_filtered,
        x='dist_to_centroid',
        y='h_index',
        color='hdbscan_cluster',
        hover_data=['display_name'],
        title='Distância ao Centroide do Cluster vs. Índice-H',
        trendline='ols' # Adiciona uma linha de tendência
    )
    fig.show()